# Creating a SQLite Database

*Define a table, insert prepared rows, and reopen a database file to verify persistence.*


## Choose a Program-Owned Database File

`sqlite3.connect(path)` opens an existing database or creates one when
the file is absent. This example owns `outputs/02_creating_sqlite_databases/store_sales.db`.
It creates the output folder before opening the database.

Relative paths use the current working directory. In the course site's
browser filesystem, the default is `/home/pyodide`, so this file appears
under `/home/pyodide/outputs/02_creating_sqlite_databases/`. Locally, run
from this chapter folder. All input rows are supplied directly below.


In [1]:
import os
import sqlite3

output_directory = os.path.join("outputs", "02_creating_sqlite_databases")
output_path = os.path.join(output_directory, "store_sales.db")
os.makedirs(output_directory, exist_ok=True)

connection = sqlite3.connect(output_path)
print("Database file:", output_path)


Database file: outputs/02_creating_sqlite_databases/store_sales.db


## Define the Table and Its Key

A table's **schema** defines its columns and rules. `CREATE TABLE`
names the columns and declares their types: `TEXT` for the product and
category, and `INTEGER` for the counts, whole-dollar prices, and month
numbers. `sale_id INTEGER PRIMARY KEY` uniquely identifies a sale;
SQLite assigns it when an inserted row omits that field. `NOT NULL`
requires values for the remaining fields.

`connection.execute()` sends one SQL statement to SQLite. The triple
quotes below are Python syntax for holding a multiline SQL string.

`DROP TABLE IF EXISTS sales` resets only this lesson's generated table.
This intentional reset makes a complete Run All reproduce five rows.
`CREATE TABLE IF NOT EXISTS` alone would not remove previously inserted data.


In [2]:
connection.execute("DROP TABLE IF EXISTS sales")
connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")
print("Fresh sales table created.")


Fresh sales table created.


## Bind Values and Insert Rows

List the destination columns explicitly. The five `?` placeholders match
the five values in each tuple. `executemany()` inserts the records and
`commit()` saves this transaction. Bind data values rather than formatting
them into SQL strings.

`SELECT * FROM sales` reads all columns from the table. `ORDER BY sale_id`
sorts the returned records by their assigned IDs. The loop prints one
result tuple per row.


In [3]:
sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Rows inserted:", len(sales_rows))
print("Saved records:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Rows inserted: 5
Saved records:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


Five records receive sale IDs 1 through 5. A full repeat recreates the generated table before inserting, so repeated runs keep five records.


## Close, Reopen, and Read the Saved File

Closing the connection releases the file. Opening the same path again
reads the saved database. `.fetchall()` returns all selected tuples, and
`.fetchone()` returns one result row. A `COUNT(*)` query produces a
one-item tuple; `[0]` reads that count.


In [4]:
connection.close()
reopened_connection = sqlite3.connect(output_path)
try:
    saved_rows = reopened_connection.execute("""
    SELECT product_name, category, quantity, unit_price, month
    FROM sales
    ORDER BY sale_id
    """).fetchall()
    row_count = reopened_connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0]
    print("Reopened records:")
    for row in saved_rows:
        print(row)
    print("Reopened row count:", row_count)
    print("Saved rows match prepared data:", saved_rows == sales_rows)
finally:
    reopened_connection.close()


Reopened records:
('Mouse', 'Accessory', 3, 25, 1)
('Keyboard', 'Accessory', 2, 45, 1)
('Monitor', 'Display', 1, 180, 2)
('Laptop Stand', 'Accessory', 4, 30, 2)
('Tablet', 'Device', 2, 220, 3)
Reopened row count: 5
Saved rows match prepared data: True


The reopened file contains five rows in the prepared order, and the comparison is `True`. `finally` closes the second connection even if reading fails.


Create a schema, bind values, commit changes, and verify persistence through a new connection. Keep resets limited to the example-owned table.

[Python sqlite3 tutorial](https://docs.python.org/3.12/library/sqlite3.html#tutorial)
